# Exploring Document Splitters and Chunkers in LangChain

## Install OpenAI, HuggingFace and LangChain dependencies

In [0]:
!pip install langchain==0.3.11
!pip install langchain-openai==0.2.12
!pip install langchain-community==0.3.11

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
  Using cached langchain-0.3.11-py3-none-any.whl (1.0 MB)
  Using cached aiohttp-3.11.11-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.6 MB)
  Using cached langchain_core-0.3.29-py3-none-any.whl (411 kB)
  Using cached pydantic-2.10.5-py3-none-any.whl (431 kB)
  Using cached PyYAML-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (751 kB)
  Using cached async_timeout-4.0.3-py3-none-any.whl (5.7 kB)
  Using cached langsmith-0.2.10-py3-none-any.whl (326 kB)
  Using cached SQLAlchemy-2.0.37-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
  Using cached langchain_text_splitters-0.3.5-py3-none-any.whl (31 kB)
  Using cached multidict-6.1.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (124 kB)
  Using cached propcache-0.2.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (205 kB)
  Using cached frozenlist-1.5.0-cp

In [0]:
# takes 2 - 5 mins to install on Colab
!pip install "unstructured[all-docs]==0.14.0"

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
  Using cached unstructured-0.14.0-py3-none-any.whl (1.9 MB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl (19 kB)
  Using cached emoji-2.14.0-py3-none-any.whl (586 kB)
  Using cached backoff-2.2.1-py3-none-any.whl (15 kB)
  Using cached langdetect-1.0.9-py3-none-any.whl
  Using cached nltk-3.9.1-py3-none-any.whl (1.5 MB)
  Using cached rapidfuzz-3.11.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
  Using cached python_iso639-2024.10.22-py3-none-any.whl (274 kB)
  Using cached python_magic-0.4.27-py2.py3-none-any.whl (13 kB)
  Using cached tabulate-0.9.0-py3-none-any.whl (35 kB)
  Using cached unstructured_client-0.28.1-py3-none-any.whl (62 kB)
  Using cached wrapt-1.17.0-cp310-cp310-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl (82 kB)
  Using cached unstructured_inference-0.7.31-py3-none-any.whl (59 kB)
  Using cac

After installing `unstructured`above remember to restart your session when it shows you the following popup, if it doesn't go to `Runtime`and `Restart Session`

![](https://i.imgur.com/UOBaotk.png)

In [0]:
# install OCR dependencies for unstructured
!sudo apt-get install tesseract-ocr
!sudo apt-get install poppler-utils




The following additional packages will be installed:
  liblept5 libopenjp2-7 libtesseract4 libwebpmux3 tesseract-ocr-eng
  tesseract-ocr-osd
The following NEW packages will be installed:
  liblept5 libopenjp2-7 libtesseract4 libwebpmux3 tesseract-ocr
  tesseract-ocr-eng tesseract-ocr-osd
0 upgraded, 7 newly installed, 0 to remove and 41 not upgraded.
Need to get 7,410 kB of archives.
After this operation, 22.2 MB of additional disk space will be used.
Do you want to continue? [Y/n] 

In [0]:
!pip install langchain-text-splitters==0.3.2
!pip install tiktoken==0.7.0
!pip install spacy
!pip install sentence-transformers==2.7.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 44.5 MB/s eta 0:00:00
  Attempting uninstall: tiktoken
    Found existing installation: tiktoken 0.8.0
    Uninstalling tiktoken-0.8.0:
      Successfully uninstalled tiktoken-0.8.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 15.0 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 3.2.1
    Uninstalling sentence-transformers-3.2.1:
      Successfully uninstalled sentence-transformers-3.2.1


## Document Splitting and Chunking

After loading documents into LangChain, you might need to transform them for optimal use in your application. One common transformation is splitting a long document into smaller segments to fit within your model's context window. LangChain provides several built-in document transformers to facilitate the splitting, combining, filtering, and manipulating of documents.

#### Process of Document Splitting:
1. **Splitting into Chunks:**
   - Break down the text into small, semantically meaningful units (typically sentences).
   
2. **Combining Chunks:**
   - Assemble these smaller units into larger chunks until they reach a predefined size. This size is determined by a specific measurement function.

3. **Creating Overlapping Chunks:**
   - Once the maximum size is reached, finalize the chunk as an independent text piece.
   - Begin a new chunk, incorporating some overlap with the previous chunk to maintain textual context.

This approach ensures that semantically related text pieces are kept together, which is crucial for maintaining the meaning and continuity of the document.


### RecursiveCharacterTextSplitter

The `RecursiveCharacterTextSplitter` is a versatile tool within LangChain for splitting text based on a list of characters. This splitter is designed to handle various requirements through adjustable parameters.

This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is ["\n\n", "\n", " ", ""]. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

#### Features and Parameters:

- **Character List:** Utilizes a specified list of characters to determine where splits should occur.
- **Chunk Size:** Allows you to set the size of each chunk, helping ensure that chunks are manageable and suit the context window of your model.
- **Overlap:** Configurable overlap between consecutive chunks to maintain context continuity across chunks.

This splitter is particularly useful for texts where precise control over the splitting criteria is needed, allowing for customized chunking strategies based on specific characters.


In [0]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

doc = """Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.
On a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.
Green Valley is not only known for its scenic beauty but also for its annual festivals. The most anticipated event is the Harvest Festival, celebrated with great enthusiasm. Locals prepare months in advance, cultivating crops and crafting goods for the occasion. The festival features a parade, various competitions, and a night market that lights up the town with vibrant colors and joyous energy.
"""



In [0]:
print(doc)

Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.
On a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.
Green Valley is not only known for its scenic beauty but also for its annual festivals. The most anticipated event is the Harvest Festival, celebrated with great enthusiasm. Locals prepare months in advance, cultivating crops and crafting goods for the occasion. The festival features a parade, various competitions, and a night market that lights up the town with vibrant colors and joy

Splitting with smaller chunk size (total characters) makes more paragraphs

In [0]:
text_splitter = RecursiveCharacterTextSplitter(
    separators= ["\n\n", "\n", " ", ""],
    chunk_size=300,
    chunk_overlap=0,
)

In [0]:
texts = text_splitter.split_text(doc)
print(len(texts)) # 5

5


In [0]:
for text in texts:
    print(text)
    print(len(text))
    print()

Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.
299

On a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream
298

of customers.
13

Green Valley is not only known for its scenic beauty but also for its annual festivals. The most anticipated event is the Harvest Festival, celebrated with great enthusiasm. Locals prepare months in advance, cultivating crops and crafting goods for the occasion. The festival features a parade,
294

various competitions, and a night market that lights up the town with vib

Splitting with larger chunk size (total characters) makes less paragraphs

In [0]:
text_splitter = RecursiveCharacterTextSplitter(
    separators= ["\n\n", "\n", " ", ""],
    chunk_size=500,
    chunk_overlap=0,
)

texts = text_splitter.split_text(doc)
print(len(texts)) # 3

3


In [0]:
for text in texts:
    print(text)
    print(len(text))
    print()

Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.
299

On a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.
312

Green Valley is not only known for its scenic beauty but also for its annual festivals. The most anticipated event is the Harvest Festival, celebrated with great enthusiasm. Locals prepare months in advance, cultivating crops and crafting goods for the occasion. The festival features a parade, various competitions, and a night market that lights up the town with vibrant colo

`chunk_overlap` helps to mitigate loss of information when context is divided between chunks especially for really small chunks

In [0]:
text_splitter = RecursiveCharacterTextSplitter(
    separators= ["\n\n", "\n", " ", ""],
    chunk_size=300,
    chunk_overlap=100,
)

texts = text_splitter.split_text(doc)
print(len(texts)) # 5

5


In [0]:
for text in texts:
    print(text)
    print(len(text))
    print()

Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.
299

On a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream
298

of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.
110

Green Valley is not only known for its scenic beauty but also for its annual festivals. The most anticipated event is the Harvest Festival, celebrated with great enthusiasm. Locals prepare months in advance, cultivating crops and crafting goods for the occasion. The festival

You can create LangChain `Document` chunks with the `create_documents` function

In [0]:
text_splitter = RecursiveCharacterTextSplitter(
    separators= ["\n\n", "\n", " ", ""],
    chunk_size=500,
    chunk_overlap=100,
)

In [0]:
docs = text_splitter.create_documents([doc])
docs

[Document(metadata={}, page_content='Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.'),
 Document(metadata={}, page_content='On a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.'),
 Document(metadata={}, page_content='Green Valley is not only known for its scenic beauty but also for its annual festivals. The most anticipated event is the Harvest Festival, celebrated with great enthusiasm. Locals prepare months in advance, cultivating crops and crafting goods for the occasion. The fes

### CharacterTextSplitter

The `CharacterTextSplitter` is a straightforward tool in LangChain for dividing text based on a specified character. It's designed to be simple yet effective, providing essential controls for customizing how text is segmented.

#### Key Features and Parameters:
- **Split Character:** By default, it uses a empty string character ("") to split the text, but this can be customized to any character you specify.
- **Chunk Size:** Allows you to define the length of each chunk in terms of the number of characters. This is useful for ensuring each piece of text is of a manageable size for processing.
- **Overlap:** You can set the amount of overlap between consecutive chunks. This helps maintain context and continuity when text is split into separate parts.

This method is the simplest among text splitting tools, focusing on character-based division and providing straightforward measures for chunk length and overlap.

To obtain the string content directly, use `.split_text`.

To create LangChain `Document` objects (e.g., for use in downstream tasks), use `.create_documents`.


In [0]:
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=500,
    chunk_overlap=200,
    is_separator_regex=False,
)

docs = text_splitter.create_documents([doc])
docs

[Document(metadata={}, page_content='Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.'),
 Document(metadata={}, page_content='On a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.'),
 Document(metadata={}, page_content='Green Valley is not only known for its scenic beauty but also for its annual festivals. The most anticipated event is the Harvest Festival, celebrated with great enthusiasm. Locals prepare months in advance, cultivating crops and crafting goods for the occasion. The fes

### Code Splitters

`RecursiveCharacterTextSplitter` includes pre-built lists of separators that are useful for splitting text in a specific programming language.

In [0]:
from langchain.text_splitter import RecursiveCharacterTextSplitter, Language

python_code = """
def hello_world():
    print("Hello, World!")
hello_world()
"""

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=50, chunk_overlap=0
)
python_docs = python_splitter.create_documents([python_code])
python_docs

[Document(metadata={}, page_content='def hello_world():\n    print("Hello, World!")'),
 Document(metadata={}, page_content='hello_world()')]

### Markdown Splitters

We might want to chunk a document based on the structure. For example, a markdown file is organized by headers. Creating chunks within specific header groups is an intuitive idea. To address this challenge, we can use MarkdownHeaderTextSplitter. This will split a markdown file by a specified set of headers.

For example, if we want to split this markdown:

```
markdown_document = """
# Team Introductions

## Management Team

Hi, this is Jim, the CEO.  
Hi, this is Joe, the CFO.

## Development Team

Hi, this is Molly, the Lead Developer.
"""
```

We can specify the headers to split on:

```
[("#", "Header 1"),
 ("##", "Header 2")]
```

And content is grouped or split by common headers:

```
Document(page_content='Hi, this is Jim, the CEO.\nHi, this is Joe, the CFO.',
metadata={'Header 1': 'Team Introductions', 'Header 2': 'Management Team'})

Document(page_content='Hi, this is Molly, the Lead Developer.',
metadata={'Header 1': 'Team Introductions', 'Header 2': 'Development Team'})
```

In [0]:
markdown_document = """
# Team Introductions

## Management Team
Hi, this is Jim, the CEO.
Hi, this is Joe, the CFO.

## Development Team
Hi, this is Molly, the Lead Developer.
"""

In [0]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_document)
md_header_splits

[Document(metadata={'Header 1': 'Team Introductions', 'Header 2': 'Management Team'}, page_content='Hi, this is Jim, the CEO.\nHi, this is Joe, the CFO.'),
 Document(metadata={'Header 1': 'Team Introductions', 'Header 2': 'Development Team'}, page_content='Hi, this is Molly, the Lead Developer.')]

By default, `MarkdownHeaderTextSplitter` strips headers being split on from the output chunk's content. This can be disabled by setting `strip_headers = False`.

In [0]:
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)
md_header_splits = markdown_splitter.split_text(markdown_document)
md_header_splits

[Document(metadata={'Header 1': 'Team Introductions', 'Header 2': 'Management Team'}, page_content='# Team Introductions  \n## Management Team\nHi, this is Jim, the CEO.\nHi, this is Joe, the CFO.'),
 Document(metadata={'Header 1': 'Team Introductions', 'Header 2': 'Development Team'}, page_content='## Development Team\nHi, this is Molly, the Lead Developer.')]

### Tokenizer based Splitting

Language models have a token limit. You should not exceed the token limit. When you split your text into chunks it is therefore a good idea to count the number of tokens. There are many tokenizers. When you count tokens in your text you should use the same tokenizer as used in the language model. Let's look at how we can chunk documents using different tokenizers



#### tiktoken splitters

[`tiktoken`](https://github.com/openai/tiktoken) is a fast BPE tokenizer created by OpenAI.

We can use tiktoken to estimate tokens used. It will probably be more accurate for the OpenAI models. We measure the `chunk_size`here based on the number of tokens typically and not the number of characters

For Open AI models, roughly 1 token = 3\4 words.

Approx: 100 tokens ~= 75 words.



We can load a [`TokenTextSplitter`](https://api.python.langchain.com/en/latest/base/langchain_text_splitters.base.TokenTextSplitter.html) splitter, which works with `tiktoken` directly and will ensure each split is smaller than chunk size in terms of the number of tokens.

In [0]:
doc = """Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.
On a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.
Green Valley is not only known for its scenic beauty but also for its annual festivals. The most anticipated event is the Harvest Festival, celebrated with great enthusiasm. Locals prepare months in advance, cultivating crops and crafting goods for the occasion. The festival features a parade, various competitions, and a night market that lights up the town with vibrant colors and joyous energy.
"""

In [0]:
from langchain_text_splitters import TokenTextSplitter

text_splitter = TokenTextSplitter(model_name='gpt-4o-mini',
                                  chunk_size=30,
                                  chunk_overlap=10)

docs = text_splitter.create_documents([doc])

In [0]:
len(docs)

10

In [0]:
docs

[Document(metadata={}, page_content='Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a'),
 Document(metadata={}, page_content=' and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering'),
 Document(metadata={}, page_content=' with an array of shops and cafes, each offering a unique taste of local flavor and culture.\nOn a typical afternoon, the town square comes alive with'),
 Document(metadata={}, page_content=' a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts'),
 Document(metadata={}, page_content=' Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts'),
 Document(metadata={}

In [0]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")
for d in docs:
  print('Words:', len(d.page_content.split(' ')),
        'Tokens:', len(enc.encode(d.page_content)),
        'Chunk:', d.page_content)

Words: 27 Tokens: 30 Chunk: Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a
Words: 28 Tokens: 30 Chunk:  and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering
Words: 27 Tokens: 30 Chunk:  with an array of shops and cafes, each offering a unique taste of local flavor and culture.
On a typical afternoon, the town square comes alive with
Words: 27 Tokens: 30 Chunk:  a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts
Words: 27 Tokens: 30 Chunk:  Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts
Words: 26 Tokens: 30 Chunk:  by. The aroma of freshly baked bread wafts from the b

Larger chunk size in terms of number of words \ tokens will create lesser chunks or paragraphs as usual

In [0]:
text_splitter = TokenTextSplitter(model_name='gpt-4o-mini',
                                  chunk_size=100,
                                  chunk_overlap=30)

docs = text_splitter.create_documents([doc])

In [0]:
len(docs)

3

In [0]:
docs

[Document(metadata={}, page_content='Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.\nOn a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone'),
 Document(metadata={}, page_content=' the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.\nGreen Valley is not only known for its scenic beauty but also for its annual festivals. The most anticipated event is the Harvest Festival, celebrate

In [0]:
enc = tiktoken.encoding_for_model("gpt-4o-mini")
for d in docs:
  print('Words:', len(d.page_content.split(' ')),
        'Tokens:', len(enc.encode(d.page_content)),
        'Chunk:', d.page_content)

Words: 88 Tokens: 100 Chunk: Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.
On a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone
Words: 86 Tokens: 100 Chunk:  the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.
Green Valley is not only known for its scenic beauty but also for its annual festivals. The most anticipated event is the Harvest Festival, celebrated with great enthusia

To implement a hard constraint on the chunk size, we can use `RecursiveCharacterTextSplitter.from_tiktoken_encoder`, where each split will be recursively split if it has a larger size and it makes the chunks more meaningful

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-4o-mini",
    chunk_size=100,
    chunk_overlap=30,
)

docs = text_splitter.create_documents([doc])

In [0]:
len(docs)

3

In [0]:
docs

[Document(metadata={}, page_content='Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.'),
 Document(metadata={}, page_content='On a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.'),
 Document(metadata={}, page_content='Green Valley is not only known for its scenic beauty but also for its annual festivals. The most anticipated event is the Harvest Festival, celebrated with great enthusiasm. Locals prepare months in advance, cultivating crops and crafting goods for the occasion. The fes

In [0]:
enc = tiktoken.encoding_for_model("gpt-4o-mini")
for d in docs:
  print('Words:', len(d.page_content.split(' ')),
        'Tokens:', len(enc.encode(d.page_content)),
        'Chunk:', d.page_content)

Words: 53 Tokens: 59 Chunk: Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.
Words: 53 Tokens: 62 Chunk: On a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.
Words: 63 Tokens: 72 Chunk: Green Valley is not only known for its scenic beauty but also for its annual festivals. The most anticipated event is the Harvest Festival, celebrated with great enthusiasm. Locals prepare months in advance, cultivating crops and crafting goods for the occasion. The festival features a parade, various 

#### spaCy

[spaCy](https://spacy.io/) is an open-source software library for advanced natural language processing, written in the programming languages Python and Cython.

LangChain implements splitters based on the [spaCy tokenizer](https://spacy.io/api/tokenizer).

In [0]:
from langchain_text_splitters import SpacyTextSplitter

text_splitter = SpacyTextSplitter(chunk_size=500,
                                  chunk_overlap=50)

docs = text_splitter.create_documents([doc])

/usr/local/lib/python3.10/dist-packages/spacy/pipeline/lemmatizer.py:211: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [0]:
len(docs)

3

In [0]:
docs

[Document(metadata={}, page_content='Welcome to Green Valley, a small town nestled in the heart of the mountains.\n\nWith its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years.\n\nThe main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.\n\n\nOn a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling.'),
 Document(metadata={}, page_content='Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by.\n\nThe aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.\n\n\nGreen Valley is not only known for its scenic beauty but also for its annual festivals.\n\nThe most anticipated event is the Harvest Festival, celebrated with great enthusiasm.\n\nLocals prepare months in advance, cultivating crops and crafting goods for the occasion.'),
 Document(metadata=

In [0]:
for d in docs:
  print('Words:', len(d.page_content.split(' ')),
        'Characters:', len(d.page_content),
        'Chunk:', d.page_content)

Words: 68 Characters: 413 Chunk: Welcome to Green Valley, a small town nestled in the heart of the mountains.

With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years.

The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.


On a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling.
Words: 72 Characters: 470 Chunk: Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by.

The aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.


Green Valley is not only known for its scenic beauty but also for its annual festivals.

The most anticipated event is the Harvest Festival, celebrated with great enthusiasm.

Locals prepare months in advance, cultivating crops and crafting goods for the occasion.
Words: 22 Characters: 135 Chunk: The festival fea

#### SentenceTransformers

The [`SentenceTransformersTokenTextSplitter`](https://api.python.langchain.com/en/latest/sentence_transformers/langchain_text_splitters.sentence_transformers.SentenceTransformersTokenTextSplitter.html) is a specialized text splitter for use with the `sentence-transformer` language models.

The default behaviour is to split the text into chunks that fit the token window of the sentence transformer model that you would like to use.

In [0]:
from langchain_text_splitters import SentenceTransformersTokenTextSplitter

splitter = SentenceTransformersTokenTextSplitter(model_name="sentence-transformers/all-mpnet-base-v2",
                                                 tokens_per_chunk=100,
                                                 chunk_overlap=30)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [0]:
docs = splitter.create_documents([doc])

In [0]:
len(docs)

3

In [0]:
docs

[Document(metadata={}, page_content='welcome to green valley, a small town nestled in the heart of the mountains. with its picturesque landscapes and vibrant community life, green valley has been a hidden gem for years. the main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture. on a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. children play near the fountain, artists display their crafts, and an old man tells stories of days'),
 Document(metadata={}, page_content='the bustling sounds of locals and visitors mingling. children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. the aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers. green valley is not only known for its scenic beauty but also for its annual festivals. the most anticipated event is the harvest festival, celebrated with g

In [0]:
for d in docs:
  print('Words:', len(d.page_content.split(' ')),
        'Characters:', len(d.page_content),
        'Chunk:', d.page_content)

Words: 88 Characters: 509 Chunk: welcome to green valley, a small town nestled in the heart of the mountains. with its picturesque landscapes and vibrant community life, green valley has been a hidden gem for years. the main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture. on a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. children play near the fountain, artists display their crafts, and an old man tells stories of days
Words: 84 Characters: 517 Chunk: the bustling sounds of locals and visitors mingling. children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. the aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers. green valley is not only known for its scenic beauty but also for its annual festivals. the most anticipated event is the harvest festival, celebrated with great enthus

### Section-based Splitting in Unstructured.io

Chunking functions in `unstructured` use metadata and document elements detected with partition functions to split a document into smaller parts for uses cases such as Retrieval Augmented Generation (RAG).

`unstructured` uses specific knowledge about each document format to partition the document into semantic units (document elements), we only need to resort to text-splitting when a single element exceeds the desired maximum chunk size. Except in that case, all chunks contain one or more whole elements, preserving the coherence of semantic units established during partitioning.

- Chunking is performed on document elements. It is a separate step performed after partitioning, on the elements produced by partitioning. (Although it can be combined with partitioning in a single step.)

- Chunking combines consecutive elements to form chunks as large as possible without exceeding the maximum chunk size.

- A single element that by itself exceeds the maximum chunk size is divided into two or more chunks using text-splitting.

- Chunking produces a sequence of `CompositeElement`, `Table`, or `TableChunk` elements. Each “chunk” is an instance of one of these three types.

Chunking Options:

The following options are available to tune chunking behaviors. These are keyword arguments that can be used in a partitioning or chunking function call. All these options have defaults and need only be specified when a non-default setting is required. Specific chunking strategies (such as “by-title”) may have additional options.

- `max_characters`: (default=500) - the hard maximum size for a chunk. No chunk will exceed this number of characters. A single element that by itself exceeds this size will be divided into two or more chunks using text-splitting.

- `new_after_n_chars`: (default=max_characters) - the “soft” maximum size for a chunk. A chunk that already exceeds this number of characters will not be extended, even if the next _element_ would fit without exceeding the specified hard maximum. This can be used in conjunction with `max_characters` to set a “preferred” size, like “I prefer chunks of around 1000 characters, but I’d rather have a chunk of 1500 (max_characters) than resort to text-splitting”. This would be specified with `(..., max_characters=1500, new_after_n_chars=1000)`.

- `overlap`: (default=0) - only when using text-splitting to break up an oversized chunk, include this number of characters from the end of the prior chunk as a prefix on the next. This can mitigate the effect of splitting the semantic unit represented by the oversized element at an arbitrary position based on text length.

- `combine_text_under_n_chars argument`: This defaults to the same value as `max_characters` such that sequential small section chunks are combined to maximally fill the chunking window to produce a logically larger chunk


There are currently two chunking strategies, `basic` and `by_title`.

The `basic` strategy combines sequential elements to maximally fill each chunk while respecting both the specified max_characters (hard-max) and new_after_n_chars (soft-max) option values.

The `by_title` chunking strategy preserves section boundaries and optionally page boundaries as well. “Preserving” here means that a single chunk will never contain text that occurred in two different sections.

In [0]:
!wget -O 'layoutparser_paper.pdf' 'http://arxiv.org/pdf/2103.15348.pdf'

--2024-12-13 18:01:30--  http://arxiv.org/pdf/2103.15348.pdf
Resolving arxiv.org (arxiv.org)... 151.101.67.42, 151.101.3.42, 151.101.195.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.67.42|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: http://arxiv.org/pdf/2103.15348 [following]
--2024-12-13 18:01:30--  http://arxiv.org/pdf/2103.15348
Reusing existing connection to arxiv.org:80.
HTTP request sent, awaiting response... 200 OK
Length: 4686220 (4.5M) [application/pdf]
Saving to: ‘layoutparser_paper.pdf’

layoutparser_paper. 100%[===================>]   4.47M  1.89MB/s    in 2.4s    

2024-12-13 18:01:33 (1.89 MB/s) - ‘layoutparser_paper.pdf’ saved [4686220/4686220]



Download nltk packages if not downloaded already

In [0]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [0]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [0]:
from langchain_community.document_loaders import UnstructuredPDFLoader

# takes 3-4 mins on Colab
loader = UnstructuredPDFLoader('./layoutparser_paper.pdf',
                               strategy='hi_res',
                               extract_images_in_pdf=False,
                               infer_table_structure=True,
                               chunking_strategy="by_title",
                               max_characters=4000,
                               new_after_n_chars=3800,
                               combine_text_under_n_chars=2000,
                               mode='elements')
data = loader.load()

In [0]:
len(data)

18

In [0]:
[doc.metadata['category'] for doc in data]

['CompositeElement',
 'CompositeElement',
 'CompositeElement',
 'CompositeElement',
 'CompositeElement',
 'Table',
 'CompositeElement',
 'CompositeElement',
 'CompositeElement',
 'Table',
 'CompositeElement',
 'CompositeElement',
 'CompositeElement',
 'CompositeElement',
 'CompositeElement',
 'CompositeElement',
 'CompositeElement',
 'CompositeElement']

In [0]:
data[0]

Document(metadata={'source': './layoutparser_paper.pdf', 'filetype': 'application/pdf', 'languages': ['eng'], 'last_modified': '2023-01-23T09:15:33', 'page_number': 1, 'orig_elements': 'eJzNWG1v2zgS/is8f2oPllbvsvPlmrbAtbvZu6LN3h62WwQUObLZSqJAUk7dYv/7PaSc1G2yCzRAgvtkc8jhvDzPzNB++3lBHfU0uAslFydskfE0l6IQUUo8iQrZJlEj1jIiypNsVUniZbtYskVPjkvuOHQ+L4TWRqqBO7Jh3fG9ntzFltRm6yDJsiSBzkF8qaTbQprWQTpqNTiv9/ZtUcbFkpXrLK7fLdlhWSd5vPLLNEni9c31fByChd1bR72P4pX6SN2bkQta/IGNVnXk9iOFrVc/L4Ivw2bim+Dw2wUNm8W7ILXuotdStYrmdCRZHiVplOXnyfokLU/y3GuP0LwYpr4h4wPxNhx99KEuUpaxhGX+1JXJXwaB1Gy0UZ9InvtzUPg28atqlSWFqKOcMooKAaPc41BUvJQNF826lPec+Koq4/JL4tMiL+P1UaZvCGaFv0y9JEfCKT1cCCTXXoxGNziWxGWR5NUDYzOwif3IPELv2H/YMxYzywRMHGH1griE4i0AZVm5Xsm8igQlWVQUKIpVXrbRKm0TUa5WeV3RPQOUJusizo4QKssyro4RuiGYNf5vqiNjO7ZiBctZCRxifCYBjxPIFfsvM4x/d+XkJPJ1XQCT0teLqLOoSWoZZUW5TsuszBtK7w8YlARonKd1nAdg5nWRrOMk4JDVZZzcIpg17lo7RV0/MHJnIROvuLFkTtgp+2VQv09tk6Qk2bnW3QflWKsNe040sjPiZlDDhj3lFvvPtZg8YOxlDwPsdODd3ip7jPS5ch3dhi4lbZGJQka8FBhIQuQR51xEIqU6q6

In [0]:
print(data[0].page_content)

1 2 0 2

n u J 1 2 ] V C . s c [

2 v 8 4 3 5 1 . 3 0 1 2 : v i X r a

LayoutParser: A Uniﬁed Toolkit for Deep Learning Based Document Image Analysis

Zejiang Shen! (4), Ruochen Zhang”, Melissa Dell?, Benjamin Charles Germain Lee*, Jacob Carlson’, and Weining Li®

1 Allen Institute for AI shannons@allenai.org 2 Brown University ruochen zhang@brown.edu 3 Harvard University {melissadell,jacob carlson}@fas.harvard.edu 4 University of Washington bcgl@cs.washington.edu 5 University of Waterloo w422li@uwaterloo.ca

Abstract. Recent advances in document image analysis (DIA) have been primarily driven by the application of neural networks. Ideally, research outcomes could be easily deployed in production and extended for further investigation. However, various factors like loosely organized codebases and sophisticated model conﬁgurations complicate the easy reuse of im- portant innovations by a wide audience. Though there have been on-going eﬀorts to improve reusability and simplify deep learn

In [0]:
data[1]

Document(metadata={'source': './layoutparser_paper.pdf', 'filetype': 'application/pdf', 'languages': ['eng'], 'last_modified': '2023-01-23T09:15:33', 'page_number': 1, 'orig_elements': 'eJy9WduO3LgR/RVinjzAUND94jcnTrAGvImBnSDAOoZBUaVuZdSSLEoz7l3k33OKVPeox72L9Sx6nrpVIimy6tSpCz/+ekUt7aibPjfV1WtxVWjfTyiqZBnkSsZUkVRpUciCfCojv6pzHV3diKsdTapSk8KcX690349V06mJjH1u1b6fp89bajbbCZIw9H3MWcQPTTVtIQ0yKx36ppt43sePURZ70Q1ehL4XfLoRj4Ik8iIWpFHqJWcFdgokV2ZvJtrxWT40X6n9aVCarv6HFxVNpKem7z7rVhnzeRj7EsN8Ly/SsMCAumlp2g9k53748cpuudvMamPP9fGKus3VJys10+ddXzV1Q1ZroR9G0g9kGN36xesgeR1ZJQ2Y+bmbdyWNfF7exERfWSNX77pp7KvZ7oeHHr5720yt3e5Tw5SRSgod51KXSss4yAPYpM5lGZJOlM5LPywuaJjQ2SFPvMIZxgnyOPVyFgRRlHv5WYmb9DzTFGGaZC9smrdEg3hPauyabvPq7ftrWSpDlVADtqX0loxQI4lpS8JMUK3sa4kHqcZJ1P0olIAiSYzYH4m+FlWvZzakaHb4qFCdavemMeLV23dvrsWkzJ0RTafbGZbaPB1t9dH8Z65LP9CKVQTNBjdryPxDjSPe3NMtH+AsdCKly1rLJM8SGacqlKWf+TIMk6gqScNA5eWgkxYeLBgmmRdb5CzPeegl1oGD1CvOPNvxzwNNmqZ5cnnQhGvQhOJnT/y0pU7QJFTr/SGfDiJKa0q0DGIiGdcFydwvM1mEsUqCqih0GlzQpwN24SjMDlxrn+

In [0]:
print(data[1].page_content)

Introduction

Deep Learning(DL)-based approaches are the state-of-the-art for a wide range of document image analysis (DIA) tasks including document image classiﬁcation [11,

2 Z. Shen et al.

37], layout detection [38, 22], table detection [26], and scene text detection [4]. A generalized learning-based framework dramatically reduces the need for the manual speciﬁcation of complicated rules, which is the status quo with traditional methods. DL has the potential to transform DIA pipelines and beneﬁt a broad spectrum of large-scale document digitization projects.

However, there are several practical diﬃculties for taking advantages of re- cent advances in DL-based methods: 1) DL models are notoriously convoluted for reuse and extension. Existing models are developed using distinct frame- works like TensorFlow [1] or PyTorch [24], and the high-level parameters can be obfuscated by implementation details [8]. It can be a time-consuming and frustrating experience to debug, reproduce, and 